# Test: "Pending" View Layer for Public Map

Creates a **read-only view** of the collector data that shows only **non-approved** features as **grey dots** — no popups, no visible fields. This is a test run before integrating into the production `manage-views-and-webmaps` notebook.

| Property | Value |
|---|---|
| Definition query | `review_status <> 'Approved' OR review_status IS NULL` |
| Capabilities | Query only (read-only) |
| Fields | All hidden |
| Popups | Disabled |
| Symbology | Small grey circle |
| Sharing | Public |

## 1 — Connect to ArcGIS Online

In [4]:
import getpass, json, os, sys, urllib3
from pathlib import Path
from arcgis.gis import GIS
from arcgis.features import FeatureLayerCollection

urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

_repo_root = os.path.abspath("..")
if _repo_root not in sys.path:
    sys.path.insert(0, _repo_root)

import views  # noqa: E402

username = input("Username: ").strip()
password = getpass.getpass("Password: ")
gis = GIS("https://www.arcgis.com", username, password, verify_cert=False)
print(f"Logged in as {gis.users.me.username}")

Setting `verify_cert` to False is a security risk, use at your own risk.


Logged in as ralouta.aiddev


## 2 — Load the Base Layer from Registry

In [5]:
_registry_path = Path("..") / "item_registry.json"
_reg = views.load_registry(_registry_path)

item_id = _reg["base_layer"]["item_id"]
flc_item = gis.content.get(item_id)
flc = FeatureLayerCollection.fromitem(flc_item)
base_layer = flc.layers[0]
source_folder = flc_item.ownerFolder
_clean = views.clean_title(flc_item.title)

public_view_id = _reg["views"]["public"]["item_id"]
public_wm_id = _reg["web_maps"]["public"]["item_id"]

print(f"Base layer: {flc_item.title}  ({base_layer.query(return_count_only=True)} features)")
print(f"Public view: {public_view_id}")
print(f"Public web map: {public_wm_id}")

Base layer: Plant Identification - AI  (3183 features)
Public view: 35525a1573714fa8aa6442a15e1b7983
Public web map: 407c7c9d103f43709d5d823045e1100c


## 3 — Create the "Pending" View Layer

Read-only view that shows everything **except** approved features. The `OR review_status IS NULL` catches newly collected features that haven't been through the workflow yet.

In [ ]:
pending_view_item, pending_lyr = views.create_view(
    gis, flc, base_layer,
    f"{_clean} - Pending",
    updateable=False,
    capabilities="Query",
    description="Read-only public view of non-approved (pending) records. Grey dots, no popups.",
    tags=f"{_clean}, Public, Pending",
    snippet=f"Pending records for {_clean} (grey dots, no popups)",
    folder=source_folder,
    def_query="review_status <> 'Approved' OR review_status IS NULL",
    field_updates=views.get_hidden_field_updates(base_layer),
    renderer=views.build_pending_renderer(),
)
print(f"✓ Pending view: {pending_view_item.title}  (ID: {pending_view_item.id})")

Created view: 32719d5405614733bd519ad0a641cc48
View ready: Plant Identification - AI - Pending  URL: https://services.arcgis.com/LG9Yn2oFqZi5PnO5/arcgis/rest/services/Plant_Identification_AI_Pending/FeatureServer/0


In [6]:
# Load existing pending view from registry (skip if you just ran the create cell above)
_reg = views.load_registry(_registry_path)
_pending_id = _reg["views"]["pending"]["item_id"]
pending_view_item, pending_lyr = views.get_view_layer(
    gis.content.get(_pending_id), gis
)
print(f"Loaded pending view: {pending_view_item.title}  (ID: {pending_view_item.id})")
print(f"Features: {pending_lyr.query(return_count_only=True)}")

Loaded pending view: Plant Identification - AI - Pending  (ID: 32719d5405614733bd519ad0a641cc48)
Features: 3183


## 5 — Share Publicly

In [5]:
pending_view_item.sharing.sharing_level = "EVERYONE"
print(f"Sharing: EVERYONE")

Sharing: EVERYONE


## 6 — Test: Query the View

In [7]:
# Re-fetch to get current properties
pending_view_item, pending_lyr = views.get_view_layer(pending_view_item, gis)

# Count features
pending_count = pending_lyr.query(return_count_only=True)

# Compare with base layer (query review_status on BASE layer, not the view)
base_count = base_layer.query(return_count_only=True)
approved_count = base_layer.query(where="review_status = 'Approved'", return_count_only=True)
expected_pending = base_count - approved_count

print(f"Base layer:    {base_count} features")
print(f"Approved:      {approved_count}")
print(f"Pending view:  {pending_count}  (expected: {expected_pending})")

# The view's definition query filters out Approved — verify by count match
assert pending_count == expected_pending, (
    f"FAIL: pending view has {pending_count} but expected {expected_pending}"
)
print(f"\n✓ Counts match — no approved features in pending view")

Base layer:    3183 features
Approved:      0
Pending view:  3183  (expected: 3183)

✓ Counts match — no approved features in pending view


## 7 — Map Preview

In [8]:
# Show the pending view on a map — verify grey dots, no popups
m = gis.map()
m.basemap.basemap = "satellite"
m.content.add(pending_lyr)
m.extent = dict(pending_lyr.properties.extent)

m

Map()

## 8 — Add Pending Layer to Existing Public Web Map

Adds the pending view as a second operational layer on the existing Public web map. The approved layer stays as-is (green dots), and the pending layer appears underneath as grey dots with popups disabled.

In [9]:
pending_renderer = views.build_pending_renderer()

# Add to Public web map
public_wm_item = views.add_pending_to_webmap(
    gis, public_wm_id, pending_view_item, pending_lyr,
    pending_renderer, _clean,
)
saved = public_wm_item.get_data()
print(f"Public map layers: {[ol['title'] for ol in saved.get('operationalLayers', [])]}")
print(f"✓ Pending layer added to Public web map")

# Add to Approver web map
approver_wm_id = _reg["web_maps"]["approver"]["item_id"]
approver_wm_item = views.add_pending_to_webmap(
    gis, approver_wm_id, pending_view_item, pending_lyr,
    pending_renderer, _clean,
)
saved = approver_wm_item.get_data()
print(f"Approver map layers: {[ol['title'] for ol in saved.get('operationalLayers', [])]}")
print(f"✓ Pending layer added to Approver web map")

Public map layers: ['Plant Identification - AI - Pending', 'Plant Identification - AI - Public']
✓ Pending layer added to Public web map
Approver map layers: ['Plant Identification - AI - Pending', 'Plant Identification - AI - Approver']
✓ Pending layer added to Approver web map


## 9 — Save to Registry

In [ ]:
_reg = views.load_registry(_registry_path)
_reg["views"]["pending"] = {"item_id": pending_view_item.id, "title": pending_view_item.title}
views.save_registry(_registry_path, _reg)
print(f"✓ Saved pending view to registry: {pending_view_item.id}")

✓ Saved pending view to registry: 32719d5405614733bd519ad0a641cc48
{
  "base_layer": {
    "item_id": "ba63860ddf264ff0910ae5cffaab6500",
    "title": "Plant Identification - AI",
    "url": "https://services.arcgis.com/LG9Yn2oFqZi5PnO5/arcgis/rest/services/Plant_Identification_AI_202604211705/FeatureServer/0",
    "schema": "Plant Identification"
  },
  "views": {
    "collector": {
      "item_id": "bc3f721e9c514d36b38947f23e1c0079",
      "title": "Plant Identification - AI - Collector"
    },
    "approver": {
      "item_id": "f2740828bd464fe6985ddba1a99e9ae8",
      "title": "Plant Identification - AI - Approver"
    },
    "public": {
      "item_id": "35525a1573714fa8aa6442a15e1b7983",
      "title": "Plant Identification - AI - Public"
    },
    "pending": {
      "item_id": "32719d5405614733bd519ad0a641cc48",
      "title": "Plant Identification - AI - Pending"
    }
  },
  "web_maps": {
    "collector": {
      "item_id": "a344064c381345d49cb4d674976d17eb",
      "title": "

## 10 — Cleanup (if test fails)

Run this cell **only** if you want to delete the pending view and remove it from the public web map. Otherwise skip.

In [ ]:
# ⚠️  ONLY run this to undo — removes pending layer from web maps and deletes the view

# 1. Remove from public web map
for _label, _wm_id in [("Public", public_wm_id), ("Approver", approver_wm_id)]:
    _wm = gis.content.get(_wm_id)
    wm_data = _wm.get_data()
    wm_data["operationalLayers"] = [
        ol for ol in wm_data["operationalLayers"]
        if ol.get("itemId") != pending_view_item.id
    ]
    _wm.update(data=json.dumps(wm_data))
    print(f"Removed pending layer from {_label} web map")

# 2. Delete the view item
pending_view_item.delete(permanent=True)
print(f"Deleted pending view: {pending_view_item.id}")

# 3. Remove from registry
_reg = views.load_registry(_registry_path)
_reg["views"].pop("pending", None)
views.save_registry(_registry_path, _reg)
print(f"Removed from registry")